In [1]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [2]:
sql = f"""
WITH payment_sessions AS (
  SELECT
    DeviceId,
    SessionId,
    MIN(Time) AS first_payment_time
  FROM `openrice-production.ORGA.PV_20260915`
  WHERE DeviceId IS NOT NULL
    AND SessionId IS NOT NULL
    AND (
      LOWER(EventAction) LIKE '%takeaway.pay%'
      OR LOWER(EventLabelRaw) LIKE '%takeaway.pay%'
    )
  GROUP BY DeviceId, SessionId
  ORDER BY first_payment_time
  LIMIT 100
)

SELECT pv.*
FROM `openrice-production.ORGA.PV_20260915` AS pv
INNER JOIN payment_sessions AS payment
  ON pv.DeviceId = payment.DeviceId
  AND pv.SessionId = payment.SessionId
ORDER BY payment.first_payment_time, pv.DeviceId, pv.SessionId, pv.Time;
"""

#Execute query
df_bq = client.query(sql).result().to_dataframe()
#df_bq

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [3]:
df_bq.to_csv("output_100.csv", index=False, encoding="utf-8-sig")


In [7]:
import pandas as pd
import json

event_mapping = {
        "or.app.start": "開啟 App",
        "or.app.resume": "重新載入 App",
        "or.qcksearch": "開啟快速搜尋",
        "or.search.quick": "搜尋餐廳",
        "or.search.layer": "搜尋頁面",
        "or.search.layer.record": "使用搜尋紀錄",
        "or.search.get-poi": "從搜尋結果進入 POI",
        "impression.poi": "瀏覽POI",
        "or.poi.get-details": "載入 POI 詳情",
        "or.poi.get-overview": "載入 POI 概覽",
        "or.poi.back": "離開 POI 頁面",
        "or.takeaway.order": "進入外賣自取點餐頁/菜單",
        "poi.emenu.get-item": "選擇食物",
        "or.takeaway.checkout": "前往結帳",
        "or.takeaway.place-order": "確認下單",
        "or.takeaway.pay": "付款",
        "user.bookmark.poi": "新增收藏",
        "or.myor.bkmpoi": "點擊「已收藏」",
        "myor.search.bkmpoi": "瀏覽收藏POI",
        "or.search.themelist.takeaway": "點擊外賣自取按鈕",

        "or.myor.orderlist": "載入「我的訂單」列表",
        "or.poi.get-photos.food": "載入餐廳食物相片",
        "or.search.layer.tips": "使用搜尋提示",
        "or.poi.get-photos.menu": "載入餐廳菜單相片",
        "or.app.openpush": "從 push notification 開啟 App",
        "myor.search.bkmpoi": "從收藏清單搜尋收藏 POI ",
        "user.review.write": "撰寫評論",
        "user.myor": "點擊「我的」",
        "or.poi.get-reviews": "載入餐廳評論",
        "or.poi.review.back": "離開餐廳評論",
        "or.search.nearby": "附近餐廳搜尋",
       "impression.sponsor.poi": "瀏覽贊助餐廳",
        "or.poi.map": "查看餐廳地圖",
        "or.myor.bkmpoi": "從收藏清單選擇餐廳",
        "or.takeaway.view-basket": "查看外賣購物籃",
        "or.deeplink.get-poi": "由 deep link 獲取 POI",
        "view.sr1.promotion": "查看搜尋結果中的廣告",
        "or.poi.photo.detail": "查看餐廳相片詳情",
        "user.share.poi": "分享餐廳",

        "or.bookmark.get-poi": "點擊收藏POI",
        "or.search.layer.search": "使用搜尋頁面分類列表",  # whatkey:下午茶，米芝蓮 ect
        "user.unbookmark.poi": "取消收藏",
        "or.search.back": "離開搜尋頁面",
        "or.takeaway.view-basket": "查看外賣購物籃",
        "or.myor.voucher": "我的優惠券",
        "or.coupon.restaurantOffer": "餐廳優惠券",
        "or.explore.reel.photo": "瀏覽短片",
        "or.sr2.reel.photo": "瀏覽短片"

}

def classify_event(event_action):
    action = str(event_action).strip().lower()
    return event_mapping.get(action, event_action)

# 時間排序，避免同一 session 的 event 順序混亂
tracking_source = df_bq.copy()
tracking_source["Time"] = pd.to_datetime(tracking_source["Time"])
tracking_source = tracking_source.sort_values(["Time"])

# 保留 EventAction 原文，並產生對應的行為判斷
tracking_source["event_action_raw"] = (
    tracking_source["EventAction"]
    .fillna("(empty event action)")
    .astype(str)
)

tracking_source["event_journey"] = (
    tracking_source["EventAction"]
    .apply(classify_event)
)

# 每個 DeviceId + SessionId 一行；timeline 與 journey 的相同 index 互相對應
behavior_tracking_100 = (
    tracking_source
    .groupby(["SessionId", "DeviceId"], as_index=False, sort=False)
    .agg(
        event_timeline=(
            "event_action_raw",
            lambda events: json.dumps(
                events.tolist(),
                ensure_ascii=False,
                indent=2
            )
        ),
        journey=(
            "event_journey",
            lambda indicators: json.dumps(
                indicators.tolist(),
                ensure_ascii=False,
                indent=2
            )
        ),
    )
)

behavior_tracking_100



,SessionId,DeviceId,event_timeline,journey
0,178112,2c1ec027-be02-4201-a8ce-09c4ca6031b2,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ..."
1,127050,1f894cab868b116f,"[\n ""or.poi.get-overview"",\n ""or.poi.get-det...","[\n ""載入 POI 概覽"",\n ""載入 POI 詳情"",\n ""載入 POI 概..."
2,925466,e5f59098851843a7,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜..."
3,119842,1daf763b-8da1-4b2f-b824-12d558a8ffda,"[\n ""or.app.resume"",\n ""impression.poi"",\n ...","[\n ""重新載入 App"",\n ""瀏覽POI"",\n ""瀏覽POI"",\n ""瀏..."
4,252253,3ea40686-e25e-4ccb-9887-64c71db37c78,"[\n ""or.app.resume"",\n ""or.qcksearch"",\n ""o...","[\n ""重新載入 App"",\n ""開啟快速搜尋"",\n ""載入 POI 詳情"",\..."
...,...,...,...,...
95,575573,8f07146afa4042f4,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""搜尋頁面"",\n ""開啟快..."
96,446503,6ef7488d-0ba3-4f3d-bdab-9e4a4355dee1,"[\n ""or.app.start"",\n ""or.qcksearch"",\n ""or...","[\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜尋餐廳"",\n ""查看搜..."
97,978762,f31a3999-2c20-4541-b438-7b81a5a67c77,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ..."
98,992237,f65a3864-4cf8-41a4-8d3b-19d23c8f35d5,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ..."


In [8]:
behavior_tracking_100.to_csv("behavior_tracking_100.csv", index=False, encoding="utf-8-sig")

Find unknown event

In [9]:
def list_unknown_events(dataframe):
    normalized_actions = (
        dataframe["EventAction"]
        .fillna("(empty event action)")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    unknown_events = (
        normalized_actions[~normalized_actions.isin(event_mapping)]
        .value_counts()
        .rename_axis("unknown_event_action")
        .reset_index(name="event_count")
    )

    return unknown_events


unknown_events = list_unknown_events(df_bq)
unknown_events

,unknown_event_action,event_count
0,impression.explore,73
1,,11
2,or.page.home,7
3,impression.photo,6
4,or.selforder.order,5
5,or.poi.photo.back,5
6,or.app.login,5
7,or.selforder.place-order,5
8,poi.emenu.photo,4
9,or.poi.get-photos.all,3


In [10]:
# locate unknown event
unknown_event_details = tracking_source[
    tracking_source["EventAction"]
    .fillna("(empty event action)")
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(unknown_events["unknown_event_action"])
][
    ["SessionId", "DeviceId", "Time", "EventAction", "EventLabelRaw"]
]

unknown_event_details

,SessionId,DeviceId,Time,EventAction,EventLabelRaw
787,573457,8e82b9d6-fc46-4090-ae71-8b74568fddc7,2026-09-15 07:31:23.700000+00:00,or.poi.get-photos.all,CityID:0;Lang:hk;Ver:7.20.5;POIID:433870;poiTy...
794,573457,8e82b9d6-fc46-4090-ae71-8b74568fddc7,2026-09-15 07:31:24.200000+00:00,impression.explore,CityID:0;Lang:hk;Ver:7.20.5;PhotoID:28349497;s...
793,573457,8e82b9d6-fc46-4090-ae71-8b74568fddc7,2026-09-15 07:31:24.200000+00:00,impression.explore,CityID:0;Lang:hk;Ver:7.20.5;PhotoID:28910882;s...
789,573457,8e82b9d6-fc46-4090-ae71-8b74568fddc7,2026-09-15 07:31:24.200000+00:00,impression.explore,CityID:0;Lang:hk;Ver:7.20.5;PhotoID:28349496;s...
791,573457,8e82b9d6-fc46-4090-ae71-8b74568fddc7,2026-09-15 07:31:24.200000+00:00,impression.explore,CityID:0;Lang:hk;Ver:7.20.5;PhotoID:28349499;s...
...,...,...,...,...,...
1983,998627,f7f9013d-24fb-4163-a67c-fea3fe84d089,2026-09-15 09:03:38.300000+00:00,,https://www.openrice.com/zh-cn/me/poi/takeaway...
1984,998627,f7f9013d-24fb-4163-a67c-fea3fe84d089,2026-09-15 09:04:10.400000+00:00,,https://www.openrice.com/zh-cn/me/poi/takeaway...
2044,370894,5c5145b0-9e5d-4355-b911-34ed7a2c0ece,2026-09-15 09:24:05.300000+00:00,impression.ad.1,CityID:0;Lang:hk;appVersion:7.20.5;sn:HK.Searc...
2045,370894,5c5145b0-9e5d-4355-b911-34ed7a2c0ece,2026-09-15 09:24:05.300000+00:00,impression.ad.1,CityID:0;Lang:hk;appVersion:7.20.5;sn:HK.Searc...


1. 定位or.takeaway.pay 
    開始倒序搜尋

2. 前面滿足四個條件（存在且順序正確，中間可夾雜其他event）：
or.takeaway.order
or.takeaway.checkout
or.takeaway.place-order


or.takeaway.pay

3. 繼續倒序搜尋，最近takeaway.order的entry source

New column: Entry Source

NA: 1. app shifting
    2. web (from google)
    3. not standard prosedure (takeaway.order -> checkout -> place-order -> payment)

In [11]:
entry_source_mapping = {
    "or.search.quick": "Search",
    "or.qcksearch": "Search",
    "or.search.nearby": "Search",
    "or.search.get-poi": "Search",
    "or.search.layer.record": "Search",
    "or.search.layer": "Search",
    "or.search.layer.tips": "Search",
    
    "or.search.themelist.takeaway": "TakeawayButton",

    "myor.search.bkmpoi": "Bookmark",
    "or.myor.bkmpoi": "Bookmark",

    "or.myor.orderlist": "OrderHistory",

    "or.deeplink.get-poi": "Deeplink",

    "or.app.openpush": "Push",

    "or.myor.voucher": "Voucher"
}

tracking_source["normalized_action"] = (
    tracking_source["EventAction"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

def find_entry_source(session_events):
    actions = session_events["normalized_action"].tolist()

    # 由最後一次 payment 開始倒序找，避免 session 內有多次 payment 時取到較早的 journey
    payment_positions = [
        index
        for index, action in enumerate(actions)
        if action == "or.takeaway.pay"
    ]

    for payment_index in reversed(payment_positions):       #place-order
        place_order_index = next(
            (   index
                for index in range(payment_index - 1, -1, -1)
                if actions[index] == "or.takeaway.place-order"),
            None
        )
        if place_order_index is None:
            continue

        checkout_index = next(          # checkout
            (   index
                for index in range(place_order_index - 1, -1, -1)
                if actions[index] == "or.takeaway.checkout"),
            None
        )
        if checkout_index is None:
            continue

        takeaway_order_index = next(           # takeaway order
            (   index
                for index in range(checkout_index - 1, -1, -1)
                if actions[index] == "or.takeaway.order"),
            None
        )
        if takeaway_order_index is None:
            continue

        # 已確認完整下單順序；在該 takeaway order 找最近entry source
        for index in range(takeaway_order_index - 1, -1, -1):
            entry_source = entry_source_mapping.get(actions[index])
            if entry_source is not None:
                return entry_source

        # 有完整下單流程，但 takeaway order 前沒有可識別入口
        return pd.NA

    # 找不到完整的 order -> checkout -> place-order -> pay 流程
    return pd.NA

session_entry_source = (
    tracking_source
    .groupby(["SessionId", "DeviceId"], sort=False)
    .apply(find_entry_source)
    .rename("entry_source")
    .reset_index()
)
behavior_tracking_100 = behavior_tracking_100.drop(
    columns=["entry_source"],
    errors="ignore"
)
behavior_tracking_100 = behavior_tracking_100.merge(
    session_entry_source,
    on=["SessionId", "DeviceId"],
    how="left"
)

behavior_tracking_100

,SessionId,DeviceId,event_timeline,journey,entry_source
0,178112,2c1ec027-be02-4201-a8ce-09c4ca6031b2,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ...",Search
1,127050,1f894cab868b116f,"[\n ""or.poi.get-overview"",\n ""or.poi.get-det...","[\n ""載入 POI 概覽"",\n ""載入 POI 詳情"",\n ""載入 POI 概...",Search
2,925466,e5f59098851843a7,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜...",Search
3,119842,1daf763b-8da1-4b2f-b824-12d558a8ffda,"[\n ""or.app.resume"",\n ""impression.poi"",\n ...","[\n ""重新載入 App"",\n ""瀏覽POI"",\n ""瀏覽POI"",\n ""瀏...",NaN
4,252253,3ea40686-e25e-4ccb-9887-64c71db37c78,"[\n ""or.app.resume"",\n ""or.qcksearch"",\n ""o...","[\n ""重新載入 App"",\n ""開啟快速搜尋"",\n ""載入 POI 詳情"",\...",Search
...,...,...,...,...,...
95,575573,8f07146afa4042f4,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""搜尋頁面"",\n ""開啟快...",Search
96,446503,6ef7488d-0ba3-4f3d-bdab-9e4a4355dee1,"[\n ""or.app.start"",\n ""or.qcksearch"",\n ""or...","[\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜尋餐廳"",\n ""查看搜...",Search
97,978762,f31a3999-2c20-4541-b438-7b81a5a67c77,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ...",Search
98,992237,f65a3864-4cf8-41a4-8d3b-19d23c8f35d5,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""重新載入 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ...",Search


In [12]:
behavior_tracking_100.to_csv("entry_source_100.csv", index=False, encoding="utf-8-sig")

In [13]:
# Entry source count and percentage; include unclassified sessions
entry_source_result = (
    behavior_tracking_100["entry_source"]
    .fillna("NA")
    .value_counts(dropna=False)
    .rename_axis("entry_source")
    .reset_index(name="count")
)

entry_source_result["%"] = (
    entry_source_result["count"]
    .div(entry_source_result["count"].sum())
    .mul(100)
    .round(2)
)

entry_source_result


,entry_source,count,%
0,Search,74,74.0
1,TakeawayButton,9,9.0
2,NA,8,8.0
3,OrderHistory,6,6.0
4,Bookmark,2,2.0
5,Deeplink,1,1.0
